# PI-KAN Hamiltonian experiment (Center-Focus, N>=7)

Workflow:
1. Train PI-KAN on a Hamiltonian-center test system.
2. Validate PDE residual (`|Lie|`).
3. Extract symbolic `Phi`.
4. Compare learned `Phi` against generated ground-truth Hamiltonian `H`.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from lab3.pi_kan.config import PIKANConfig
from lab3.pi_kan.train import train
from lab3.pi_kan.symbolic_extract import (
    extract_symbolic_polynomial,
    taylor_series_from_model,
    save_symbolic_expression,
    hamiltonian_sympy,
    phi_h_comparison_metrics,
)


In [ ]:
# Uses new defaults from config.py
cfg = PIKANConfig()
cfg


In [ ]:
result = train(cfg)
model = result['model']
system = result['system']
logs = result['logs']
print('Device used:', result['device'])
print('Checkpoint:', result['checkpoint_path'])


In [ ]:
# Plot losses from logs
loss_total = np.array([r['loss_total'] for r in logs])
loss_pde = np.array([r['loss_pde'] for r in logs])
loss_anchor = np.array([r['loss_anchor'] for r in logs])
loss_origin = np.array([r['loss_origin'] for r in logs])

plt.figure(figsize=(8,4))
plt.semilogy(loss_total, label='total')
plt.semilogy(loss_pde, label='pde')
plt.semilogy(loss_anchor, label='anchor')
plt.semilogy(loss_origin, label='origin')
plt.legend()
plt.grid(alpha=0.3)
plt.title('Training losses (Hamiltonian run)')
plt.xlabel('log entries')
plt.show()


In [ ]:
# Grid validation for Lie residual
model.eval()
R = cfg.domain_radius
n = cfg.eval_grid_n
x = torch.linspace(-R, R, n, device=next(model.parameters()).device)
y = torch.linspace(-R, R, n, device=next(model.parameters()).device)
X, Y = torch.meshgrid(x, y, indexing='xy')
pts = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=-1).requires_grad_(True)
Phi = model(pts).squeeze(-1)
grad = torch.autograd.grad(Phi, pts, grad_outputs=torch.ones_like(Phi), create_graph=False)[0]
dx, dy = system.vector_field(pts[:,0], pts[:,1])
Lie = grad[:,0]*dx + grad[:,1]*dy

phi_img = Phi.detach().reshape(n, n).cpu().numpy()
lie_img = Lie.detach().abs().reshape(n, n).cpu().numpy()

print('mean |Lie|:', float(lie_img.mean()))
print('max  |Lie|:', float(lie_img.max()))

fig, ax = plt.subplots(1,2, figsize=(10,4))
cs = ax[0].contourf(X.detach().cpu().numpy(), Y.detach().cpu().numpy(), phi_img, levels=25)
fig.colorbar(cs, ax=ax[0])
ax[0].set_title('Phi(x,y)')
ax[0].set_aspect('equal')

im = ax[1].imshow(lie_img.T, origin='lower', extent=[-R,R,-R,R], aspect='equal')
fig.colorbar(im, ax=ax[1])
ax[1].set_title('|Lie derivative|')
plt.tight_layout()
plt.show()


In [ ]:
# Symbolic extraction
if cfg.basis_type in ('chebyshev', 'legendre'):
    phi_sym = extract_symbolic_polynomial(model, cfg)
else:
    phi_sym = taylor_series_from_model(model, cfg, order=cfg.taylor_order)

h_sym = hamiltonian_sympy(system)
delta_sym = (phi_sym - h_sym).expand()

print('Phi_sym:')
print(phi_sym)
print('
H_sym (ground truth):')
print(h_sym)
print('
Delta = Phi_sym - H_sym:')
print(delta_sym)

txt_path, tex_path = save_symbolic_expression(phi_sym, cfg.output_dir)
print('
Saved symbolic expression:')
print(' -', txt_path)
print(' -', tex_path)


In [ ]:
metrics = phi_h_comparison_metrics(model, system, radius=cfg.domain_radius, grid_n=cfg.eval_grid_n)
print('Phi vs H metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.6e}')
